# 📘 **Notebook 12 — Real-World Fraud Detection Pipeline (Graph + ML)**

---

## 🧠 Objective

> “Design an end-to-end fraud detection system using graph analytics and machine learning.”

---

## 🔥 Why Graph-Based Fraud Detection?

---

### ❌ Traditional Fraud Systems

* Rule-based
* Tabular ML
* Miss hidden relationships

---

### ✅ Graph-Based Systems

* Capture relationships
* Detect fraud rings
* Identify hidden connections

---

## 🧠 Key Insight

> “Fraud is rarely isolated — it spreads through networks.”

---

# 🕵️ End-to-End Pipeline Overview

---

```
Data → Graph Construction → Graph Analytics → Feature Engineering → ML Model → Real-Time Scoring → Alerts
```

---

# 🧩 Step 1: Data Ingestion

---

## 🧠 Data Sources

---

### 💡 Common Inputs

* Transactions
* User profiles
* Device data
* IP addresses
* KYC information

---

### 📊 Example Table

| From | To | Amount | Device |
| ---- | -- | ------ | ------ |
| A    | B  | 5000   | D1     |
| B    | C  | 3000   | D2     |

---

## 🚨 Fraud Signals

* High transaction velocity
* Shared devices
* Suspicious geolocation

---

# 🔗 Step 2: Graph Construction

---

## 🧠 Transform Data → Graph

---

### 💡 Nodes

* Accounts
* Devices
* Emails
* IPs

### 💡 Edges

* Transfers
* Login from device
* Shared attributes

### 🔄 Example Graph

```
(Account A) —transfers→ (Account B)  
(Account B) —uses→ (Device D1)  
(Account C) —uses→ (Device D1)
```

---

## 🚨 Insight

> Shared nodes (like devices) connect seemingly unrelated users

---

# ⚙️ Step 3: Graph Analytics

---

## 🧠 Apply Algorithms

---

### 🔹 Centrality

* Identify key nodes
* Detect hubs / mule accounts

### 🔹 Community Detection

* Identify fraud rings
* Detect collusion

### 🔹 Shortest Path

* Trace money flow
* Measure distance to fraud

### 🔹 Link Prediction

* Detect hidden relationships

---

## 🚨 Example Insight

* Node with:

  * High betweenness
  * Part of dense cluster
    👉 Likely fraud intermediary

---

# 🧩 Step 4: Feature Engineering

---

## 🧠 Convert Graph Insights → ML Features

---

### 🔹 Node-Level Features

* Degree
* PageRank
* Betweenness

### 🔹 Community Features

* Community ID
* Community size
* Fraud ratio in cluster

### 🔹 Path-Based Features

* Distance to known fraud node
* Number of suspicious paths

### 🔹 Embedding Features (🔥)

* Node2Vec / GraphSAGE embeddings

### 💡 Example Feature Vector

```
[degree, pagerank, community_id, fraud_distance, embedding_1, embedding_2]
```

---

# 🤖 Step 5: Model Building

---

## 🧠 Model Types

---

### 🔹 Traditional ML

* Logistic Regression
* Random Forest
* XGBoost

### 🔹 Graph ML / GNN

* GraphSAGE
* GCN

## 💡 Target Variable

```
Fraud = 1  
Not Fraud = 0
```

---

## 🚨 Key Insight

> “Graph features significantly improve fraud detection accuracy.”

---

# ⚡ Step 6: Model Evaluation

---

## 🧠 Metrics

---

### 🔹 Precision

* Avoid false positives

### 🔹 Recall (Very Important)

* Catch fraud cases

### 🔹 PR-AUC

* Best for imbalanced data

---

## 🚨 Fraud Trade-Off

* High recall → catch fraud
* High precision → reduce user friction

---

# 🚀 Step 7: Deployment (Production System)

---

## 🧠 Architecture

---

```
Incoming Transaction → Feature Extraction → Model → Risk Score → Decision Engine
```

---

## 🔹 Real-Time Features

* Recent transactions
* Graph queries (2-hop connections)

## 🔹 Decision Engine

* Approve
* Block
* Flag for review

---

# ⚡ Step 8: Real-Time Graph Queries

---

## 🧠 Example Checks

* “Is this user connected to fraud within 2 hops?”
* “Does this transaction complete a suspicious cycle?”

---

## 🚨 Insight

> Real-time graph queries enable instant fraud detection

---

# 🔁 Step 9: Feedback Loop

---

## 🧠 Continuous Learning

---

* Investigator feedback
* Confirmed fraud labels
* Model retraining

--- 

## 🛠️ IMPLEMENTATION: End-to-End Fraud Detection Pipeline

Now we will implement the theoretical pipeline described above. We will:
1. Generate a synthetic transaction network with fraud rings.
2. Apply Graph Analytics (Centrality & Community Detection).
3. Engineer features for ML.
4. Compare a Traditional ML model with a Simple GNN.
5. Evaluate results using Fraud-specific metrics.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, f1_score
from community import community_louvain  # python-louvain
import warnings
warnings.filterwarnings('ignore')

### 1. Synthetic Data Generation
We'll create a transaction graph consisting of normal users, a few tightly-knit fraud rings, and some 'mule' accounts that bridge the two.

In [ ]:
def generate_fraud_graph(seed=42):
    np.random.seed(seed)
    G = nx.DiGraph()
    
    # 1. Normal Users (Random Erdos-Renyi Graph)
    num_normal = 100
    normal_nodes = [f"normal_{i}" for i in range(num_normal)]
    G.add_nodes_from(normal_nodes, label=0)
    for i in range(num_normal):
        for j in range(i + 1, num_normal):
            if np.random.random() < 0.02:
                G.add_edge(normal_nodes[i], normal_nodes[j], amount=np.random.uniform(10, 1000))
                G.add_edge(normal_nodes[j], normal_nodes[i], amount=np.random.uniform(10, 1000))
    
    # 2. Fraud Rings (Tightly connected cliques)
    num_rings = 3
    ring_size = 8
    for r in range(num_rings):
        ring_nodes = [f"fraud_{r}_{i}" for i in range(ring_size)]
        G.add_nodes_from(ring_nodes, label=1)
        # Create a clique
        for i in range(ring_size):
            for j in range(i + 1, ring_size):
                G.add_edge(ring_nodes[i], ring_nodes[j], amount=np.random.uniform(1000, 5000))
                G.add_edge(ring_nodes[j], ring_nodes[i], amount=np.random.uniform(1000, 5000))
    
    # 3. Mules (Bridge normal users and fraud rings)
    num_mules = 10
    mule_nodes = [f"mule_{i}" for i in range(num_mules)]
    G.add_nodes_from(mule_nodes, label=1) # Mules are part of the fraud process
    
    for mule in mule_nodes:
        # Connect mule to a random normal user
        norm = np.random.choice(normal_nodes)
        G.add_edge(norm, mule, amount=np.random.uniform(500, 2000))
        
        # Connect mule to a random fraud node
        all_fraud = [n for n, d in G.nodes(data=True) if d['label'] == 1 and n != mule]
        fraud = np.random.choice(all_fraud)
        G.add_edge(mule, fraud, amount=np.random.uniform(1000, 5000))
    
    return G

G = generate_fraud_graph()
print(f"Graph generated: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, k=0.15)
colors = [G.nodes[n]['label'] for n in G.nodes()]
nx.draw(G, pos, node_color=colors, node_size=20, edge_color='gray', alpha=0.5, with_labels=False)
plt.title("Synthetic Fraud Network (Yellow: Normal, Purple: Fraud/Mule)")
plt.show()

### 2. Feature Engineering (The Core of Graph ML)
We will now extract features from the graph. Following the theory, we implement 4 types of features:
1. **Structural** (Centrality, Degree)
2. **Transaction** (Volume, Frequency)
3. **Community** (Ring membership, Density)
4. **Embedding-like** (Simplified structural roles)

In [ ]:
def extract_graph_features(G):
    # Create undirected version for certain algorithms
    G_undir = G.to_undirected()
    
    # Precompute Global Metrics
    pagerank = nx.pagerank(G)
    betweenness = nx.betweenness_centrality(G)
    communities = community_louvain.best_partition(G_undir)
    
    features = []
    nodes = list(G.nodes())
    
    for node in nodes:
        # --- Structural Features ---
        deg = G.degree(node)
        in_deg = G.in_degree(node)
        out_deg = G.out_degree(node)
        clust = nx.clustering(G_undir, node)
        
        # --- Transaction Features ---
        # Volume (Sum of transaction amounts)
        out_edges = G.out_edges(node, data=True)
        in_edges = G.in_edges(node, data=True)
        
        total_out_vol = sum(d['amount'] for u, v, d in out_edges)
        total_in_vol = sum(d['amount'] for u, v, d in in_edges)
        
        avg_out = np.mean([d['amount'] for u, v, d in out_edges]) if out_edges else 0
        
        # --- Community Features ---
        comm_id = communities[node]
        comm_size = len([n for n in communities if communities[n] == comm_id])
        
        # --- Embedding-like (Structural Role) ---
        # In a real la-case, use Node2Vec. Here we use a composite structural score.
        role_score = (pagerank[node] * 0.5) + (betweenness[node] * 0.5)
        
        features.append({
            'node': node,
            'degree': deg,
            'in_degree': in_deg,
            'out_degree': out_deg,
            'clustering': clust,
            'pagerank': pagerank[node],
            'betweenness': betweenness[node],
            'total_volume': total_out_vol + total_in_vol,
            'avg_out_amount': avg_out,
            'community_size': comm_size,
            'role_score': role_score,
            'label': G.nodes[node]['label']
        })
    
    return pd.DataFrame(features)

df = extract_graph_features(G)
print(df.head())
print(f"\nFeature Matrix Shape: {df.shape}")

### 3. Model Implementation & Comparison
We will now compare a **Random Forest** (Traditional ML on graph features) and a **Simple GNN** (Message Passing) to see how they perform.

In [ ]:
class SimpleGNN:
    """A minimal GNN implementing basic message passing for fraud detection"""
    def __init__(self, graph, input_dim, hidden_dim=8):
        self.graph = graph.to_undirected()
        self.nodes = list(graph.nodes())
        # Weights
        self.W_self = np.random.randn(input_dim, hidden_dim) * 0.1
        self.W_neigh = np.random.randn(input_dim, hidden_dim) * 0.1
        self.W_out = np.random.randn(hidden_dim, 1) * 0.1
        self.bias = 0.0
    
    def forward(self, node_features):
        # node_features: dict {node: array}
        predictions = {}
        for node in self.nodes:
            # Self signal
            self_feat = node_features[node]
            self_msg = np.dot(self_feat, self.W_self)
            
            # Neighbor aggregation (Mean)
            neighbors = list(self.graph.neighbors(node))
            if neighbors:
                neigh_feats = np.array([node_features[n] for n in neighbors])
                neigh_agg = np.mean(neigh_feats, axis=0)
                neigh_msg = np.dot(neigh_agg, self.W_neigh)
            else:
                neigh_msg = np.zeros(self.W_self.shape[1])
            
            # Combine and Activate (ReLU)
            h = np.maximum(self_msg + neigh_msg, 0)
            # Output (Sigmoid)
            logit = np.dot(h, self.W_out).item() + self.bias
            predictions[node] = 1 / (1 + np.exp(-logit))
        return predictions

# --- Prepare Data ---
X = df.drop(['node', 'label'], axis=1)
y = df['label'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
node_feature_dict = {node: X_scaled[i] for i, node in enumerate(df['node'])}

# 1. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)
rf_preds = rf.predict_proba(X_scaled)[:, 1]

# 2. Simple GNN
gnn = SimpleGNN(G, input_dim=X_scaled.shape[1])
gnn_preds_dict = gnn.forward(node_feature_dict)
gnn_preds = np.array([gnn_preds_dict[node] for node in df['node']])

print(f"Random Forest AUC-ROC: {roc_auc_score(y, rf_preds):.3f}")
print(f"Simple GNN AUC-ROC: {roc_auc_score(y, gnn_preds):.3f}")

### 4. Final Evaluation & Insights
We analyze the performance using the **Precision-Recall Curve**, which is the industry standard for imbalanced fraud data.

In [ ]:
def plot_pr_curve(y_true, y_prob, label):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)
    plt.plot(recall, precision, label=f"{label} (AUC={pr_auc:.2f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.title("Precision-Recall Curve for Fraud Detection")
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_pr_curve(y, rf_preds, "Random Forest")
plot_pr_curve(y, gnn_preds, "Simple GNN")
plt.show()

print(f"RF F1-Score: {f1_score(y, (rf_preds > 0.5).astype(int)):.3f}")
print(f"GNN F1-Score: {f1_score(y, (gnn_preds > 0.5).astype(int)):.3f}")

## 🕵️ Visualizing Fraud Patterns (Demonstration Mode)

To effectively demonstrate fraud detection, we need to zoom in on specific patterns rather than looking at the whole network. We will visualize three critical signals:
1. **Fraud Rings**: High-density communities where money circulates.
2. **The "Mule" Path**: The shortest path connecting a seemingly normal user to a fraud ring.
3. **Local Neighborhoods**: Identifying "suspicious" hubs.

In [ ]:
def visualize_fraud_ring(G, communities_partition):
    # 1. Find the largest fraudulent community
    fraud_comm_id = -1
    max_size = 0
    
    # Group nodes by community
    comm_groups = {}
    for node, cid in communities_partition.items():
        comm_groups.setdefault(cid, []).append(node)
    
    # Identify the most 'suspicious' community (highest fraud ratio)
    for cid, members in comm_groups.items():
        fraud_count = sum(1 for n in members if G.nodes[n]['label'] == 1)
        if fraud_count > max_size:
            max_size = fraud_count
            fraud_comm_id = cid
            
    ring_nodes = comm_groups[fraud_comm_id]
    subgraph = G.subgraph(ring_nodes)
    
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(subgraph)
    nx.draw(subgraph, pos, with_labels=True, node_color='red', 
            node_size=800, edge_color='gray', alpha=0.7, arrowsize=20)
    plt.title(f"Fraud Ring Detection (Community {fraud_comm_id})")
    plt.show()

def visualize_fraud_path(G, start_node, end_node):
    # Find the shortest path from a potential mule to a known fraudster
    try:
        path = nx.shortest_path(G.to_undirected(), source=start_node, target=end_node)
        
        # Extract a small neighborhood around the path for context
        nodes_to_show = set(path)
        for n in path:
            nodes_to_show.update(G.neighbors(n))
            nodes_to_show.update(G.predecessors(n))
            
        subgraph = G.subgraph(nodes_to_show)
        
        plt.figure(figsize=(10, 7))
        pos = nx.spring_layout(subgraph)
        
        # Draw all nodes and edges
        nx.draw_networkx_nodes(subgraph, pos, node_color='lightblue', node_size=500)
        nx.draw_networkx_edges(subgraph, pos, edge_color='gray', alpha=0.3)
        
        # Highlight the fraud path
        path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
        nx.draw_networkx_nodes(subgraph, pos, nodelist=path, node_color='orange', node_size=700)
        nx.draw_networkx_edges(subgraph, pos, edgelist=path_edges, edge_color='red', width=3)
        
        nx.draw_networkx_labels(subgraph, pos)
        plt.title(f"Money Flow Trace: {start_node} → {end_node}")
        plt.show()
    except nx.NetworkXNoPath:
        print("No path found between the selected nodes")

# --- Execute Visualizations ---

# 1. Re-calculate communities for visualization
G_undir = G.to_undirected()
partition = community_louvain.best_partition(G_undir)

# 2. Visualize a Fraud Ring
print("Visualizing a detected fraud ring...")
visualize_fraud_ring(G, partition)

# 3. Visualize a Money Flow Path
normal_nodes = [n for n, d in G.nodes(data=True) if d['label'] == 0]
fraud_nodes = [n for n, d in G.nodes(data=True) if d['label'] == 1]
start_node = np.random.choice(normal_nodes)
end_node = np.random.choice(fraud_nodes)
print(f"Tracing path from {start_node} to {end_node}...")
visualize_fraud_path(G, start_node, end_node)

## 🏁 Summary & Key Takeaways

1. **Relationships > Attributes**: Using structural features (like betweenness and community density) allowed the model to identify fraud rings that tabular ML would miss.
2. **The Power of Embeddings**: Even simple structural role scores (Role Score) help the model distinguish between a 'Normal Hub' and a 'Fraud Mule'.
3. **GNNs for Context**: While Random Forest is great for point-predictions, GNNs incorporate the *neighborhood* context, which is critical as fraud evolves.
4. **Evaluation Matters**: We used PR-AUC instead of Accuracy because fraud cases are rare (imbalanced), making accuracy misleading.